In [13]:
# importing needed packages
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy import stats
# import cartopy.crs as ccrs
# import cartopy.feature as cfeature
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [14]:
# read in the cleaned CSV with data from central valley stations
path = 'E:/Central Valley Fog/CV_rows_codes_incl.csv'
if os.path.exists(path):
    CV_rows = pd.read_csv(path, low_memory = False)
else:
    path = '/Volumes/disk1/Central Valley Fog/CV_rows_codes_incl.csv'
    CV_rows = pd.read_csv(path, low_memory = False)

print(CV_rows)

           STATION_ID  LATITUDE  LONGITUDE                 DATE REPORT_TYPE  \
0         USM00074917   35.1170  -119.3000  1941-12-01 00:00:00         SAO   
1         USM00074917   35.1170  -119.3000  1941-12-01 01:00:00         SAO   
2         USM00074917   35.1170  -119.3000  1941-12-01 02:00:00         SAO   
3         USM00074917   35.1170  -119.3000  1941-12-01 03:00:00         SAO   
4         USM00074917   35.1170  -119.3000  1941-12-01 04:00:00         SAO   
...               ...       ...        ...                  ...         ...   
12877763  USW00024257   40.5175  -122.2986  2026-06-20 04:53:00       FM-15   
12877764  USW00024257   40.5175  -122.2986  2026-06-20 05:53:00       FM-15   
12877765  USW00024257   40.5175  -122.2986  2026-06-20 06:53:00       FM-15   
12877766  USW00024257   40.5175  -122.2986  2026-06-20 07:53:00       FM-15   
12877767  USW00024257   40.5175  -122.2986  2026-06-20 08:53:00       FM-15   

          HourlyPrecipitation HourlyPresentWeatherT

In [15]:
print("CV_rows:", CV_rows.shape) 

CV_rows: (12877768, 15)


In [16]:
CV_rows1 = CV_rows[CV_rows["HourlyPrecipitation"] < 0.03]

print("CV_rows1:", CV_rows1.shape)

CV_rows1: (12476729, 15)


In [17]:
# Unique codes in REPORT_TYPE
report_type_codes = sorted(CV_rows1["REPORT_TYPE"].dropna().astype(str).unique())

# Unique sky-condition codes (e.g., CLR, FEW, SCT, BKN, OVC) parsed from strings like "BKN:06"
hourly_sky_codes = sorted(
    CV_rows1["HourlySkyConditions"]
    .dropna()
    .astype(str)
    .str.findall(r"([A-Z]+)(?=:)")
    .explode()
    .dropna()
    .unique()
)

#find unique dailyweather codes (e.g., CLR, FEW, SCT, BKN, OVC) parsed from strings like "BKN:06"
hourly_weather_codes = sorted(
    CV_rows1["HourlyPresentWeatherType"]
    .dropna()
    .astype(str)
    .str.findall(r"([A-Z]+)(?=:)")
    .explode()
    .dropna()
    .unique()
)

print(f"REPORT_TYPE unique codes ({len(report_type_codes)}):")
print(report_type_codes)

print(f"\nHourlySkyConditions unique codes ({len(hourly_sky_codes)}):")
print(hourly_sky_codes)

print(f"\nHourlyPresentWeatherType unique codes ({len(hourly_weather_codes)}):")
print(hourly_weather_codes)

REPORT_TYPE unique codes (10):
['AUTO', 'FM-12', 'FM-15', 'FM-16', 'SAO', 'SAOSP', 'SMARS', 'SY-MT', 'SYSA', 'WBO_F']

HourlySkyConditions unique codes (5):
['BKN', 'CLR', 'FEW', 'OVC', 'SCT']

HourlyPresentWeatherType unique codes (33):
['BCBR', 'BCFG', 'BLDU', 'BR', 'DS', 'DU', 'DZ', 'FC', 'FG', 'FU', 'FZFG', 'FZRA', 'GR', 'GS', 'HZ', 'IC', 'MIBR', 'MIFG', 'PL', 'PRFG', 'RA', 'SHRA', 'SN', 'SQ', 'SS', 'TSRA', 'UP', 'VCBLDU', 'VCFG', 'VCFU', 'VCRA', 'VCSHRA', 'VCSN']


REPORT_TYPE unique codes + meanings:\
'AUTO' - AUTO\
'FM-12' - FM12-SYNOP-fixed-land-stn\
'FM-15' - FM15-METAR-Aviation-routine-wx\
'FM-16' - FM16-SPECI_Aviation-selected-special-wx\
'SAO' - SAO-Airways-incl-record-specials\
'SAOSP' - SAOSP-Airways-special-excl-record-specials\
'SMARS' - SMARS-Supp-airways-stn\
'SY-MT' - SYMT_Synop-and-METAR-merged\
'SYSA' - SYSA-Synop-and-airways-merged\
'WBO_F' - WBO(?) 

HourlySkyConditions unique codes + meanings:\
'BKN' - broken clouds\
'CLR' - clear \
'FEW' - few clouds \
'OVC' - overcast \
'SCT' - scattered clouds 

HourlyPresentWeatherConditions unique codes + meanings:\
'BCBR' - patches mist\
'BCFG' - patches fog\
'BLDU' - blowing widespread dust\
'BR' - mist\
'DS' - dust storm\
'DU' - widespread dust\
'DZ' drizzle\
'FC' - funnel cloud, waterspout, or tornado\
'FG' - fog\
'FU' - smoke\
'FZFG' - freezing fog\
'FZRA' - freezing rain\
'GR' - hail\
'GS' - small hail and/or snow pellets\
'HZ' - haze\
'IC' - ice crystals\
'MIBR' - shallow mist\
'MIFG' - shallow fog\
'PL' - ice pellets\
'PRFG' - partial fog\
'RA' - rain\
'SHRA' - showers rain\
'SN' - snow\
'SQ' - squalls\
'SS' - sandstorm\
'TSRA' - thunderstorm rain\
'UP'- unknown precipitation \
'VCBLDU' - vicinity blowing widespread dust\
'VCFG' - vicinity fog\
'VCFU' - vicinity smoke\
'VCRA' - vicinity rain\
'VCSHRA' - vicinity showers rain
'VCSN' - vicinity snow

In [18]:
# Use whichever weather column exists
weather_col = "HourlyPresentWeatherConditions" if "HourlyPresentWeatherConditions" in CV_rows1.columns else "HourlyPresentWeatherType"

# Fill HourlyVisibility with 0.9 only where weather contains "FG" and visibility is NaN
fg_nan_mask = (
    CV_rows1[weather_col].fillna("").astype(str).str.contains("FG", regex=False)
    & CV_rows1["HourlyVisibility"].isna()
)

CV_rows1.loc[fg_nan_mask, "HourlyVisibility"] = 0.9

print(f"Rows updated: {fg_nan_mask.sum()}")

Rows updated: 214799


In [19]:
# # save cleaned dataframe
# CV_rows1.to_csv('CV_rows_codes.csv', index=False)

In [20]:
foggy_rows = CV_rows1[CV_rows1["HourlyVisibility"] < 1.000]

In [21]:
# Unique codes in REPORT_TYPE
report_type_codes_fog = sorted(CV_rows1["REPORT_TYPE"].dropna().astype(str).unique())

# Unique sky-condition codes (e.g., CLR, FEW, SCT, BKN, OVC) parsed from strings like "BKN:06"
hourly_sky_codes_fog = sorted(
    foggy_rows["HourlySkyConditions"]
    .dropna()
    .astype(str)
    .str.findall(r"([A-Z]+)(?=:)")
    .explode()
    .dropna()
    .unique()
)

#find unique dailyweather codes (e.g., CLR, FEW, SCT, BKN, OVC) parsed from strings like "BKN:06"
hourly_weather_codes_fog = sorted(
    foggy_rows["HourlyPresentWeatherType"]
    .dropna()
    .astype(str)
    .str.findall(r"([A-Z]+)(?=:)")
    .explode()
    .dropna()
    .unique()
)

print(f"REPORT_TYPE unique codes ({len(report_type_codes_fog)}):")
print(report_type_codes_fog)

print(f"\nHourlySkyConditions unique codes ({len(hourly_sky_codes_fog)}):")
print(hourly_sky_codes_fog)

print(f"\nHourlyPresentWeatherType unique codes ({len(hourly_weather_codes_fog)}):")
print(hourly_weather_codes_fog)

REPORT_TYPE unique codes (10):
['AUTO', 'FM-12', 'FM-15', 'FM-16', 'SAO', 'SAOSP', 'SMARS', 'SY-MT', 'SYSA', 'WBO_F']

HourlySkyConditions unique codes (5):
['BKN', 'CLR', 'FEW', 'OVC', 'SCT']

HourlyPresentWeatherType unique codes (21):
['BCBR', 'BCFG', 'BLDU', 'BR', 'DS', 'DZ', 'FG', 'FU', 'FZFG', 'FZRA', 'HZ', 'MIBR', 'MIFG', 'PL', 'PRFG', 'RA', 'SN', 'SQ', 'SS', 'UP', 'VCFG']


In [22]:
fg_in_foggy = foggy_rows["HourlyPresentWeatherType"].fillna("").astype(str).str.contains("FG", regex=False)

print("foggy_rows with 'FG' in HourlyPresentWeatherType:", fg_in_foggy.sum())
print("foggy_rows without 'FG' in HourlyPresentWeatherType:", (~fg_in_foggy).sum())

foggy_rows with 'FG' in HourlyPresentWeatherType: 461073
foggy_rows without 'FG' in HourlyPresentWeatherType: 21565


In [24]:
non_fg_foggy = foggy_rows[~fg_in_foggy]
print(non_fg_foggy)

           STATION_ID  LATITUDE  LONGITUDE                 DATE REPORT_TYPE  \
1231      USM00074917   35.1170  -119.3000  1942-01-21 07:00:00         SAO   
1232      USM00074917   35.1170  -119.3000  1942-01-21 08:00:00         SAO   
1233      USM00074917   35.1170  -119.3000  1942-01-21 09:00:00         SAO   
1236      USM00074917   35.1170  -119.3000  1942-01-21 12:00:00         SAO   
1441      USM00074917   35.1170  -119.3000  1942-01-30 01:00:00         SAO   
...               ...       ...        ...                  ...         ...   
12866119  USW00024257   40.5175  -122.2986  2025-07-14 07:53:00       FM-15   
12866120  USW00024257   40.5175  -122.2986  2025-07-14 08:53:00       FM-15   
12866121  USW00024257   40.5175  -122.2986  2025-07-14 09:08:00       FM-16   
12866122  USW00024257   40.5175  -122.2986  2025-07-14 09:53:00       FM-15   
12866123  USW00024257   40.5175  -122.2986  2025-07-14 10:00:00       FM-12   

          HourlyPrecipitation           HourlyPrese

In [27]:
non_fg_weather_codes = sorted(
    non_fg_foggy["HourlyPresentWeatherType"]
    .dropna()
    .astype(str)
    .str.findall(r"([A-Z]+)(?=:)")
    .explode()
    .dropna()
    .unique()
)

print(f"Unique HourlyPresentWeatherType codes in non_fg_foggy ({len(non_fg_weather_codes)}):")
print(non_fg_weather_codes)

Unique HourlyPresentWeatherType codes in non_fg_foggy (12):
['BLDU', 'BR', 'DS', 'DZ', 'FU', 'HZ', 'MIBR', 'PL', 'RA', 'SN', 'SS', 'UP']


'BLDU' - blowing dust\
'BR' - mist\
'DS' - dust storm\
'DZ' - drizzle\
'FU' - smoke\
'HZ' - haze\
'MIBR' - shallow mist\
'PL' -  ice pellets\
'RA' - rain\
'SN' - snow\
'SS' - sandstorm\
'UP' - unknown precipitation

In [25]:
# # Keep only rows with FG code in foggy_rows
# foggy_rows = foggy_rows[fg_in_foggy].copy()

# print(f"Removed rows without FG: {(~fg_in_foggy).sum()}")
# print("foggy_rows (FG only):", foggy_rows.shape)

Removed rows without FG: 21565
foggy_rows (FG only): (461073, 15)


In [26]:
# foggy_rows.to_csv('E:/Central Valley Fog/FG_rows.csv', index=False)
# print("Saved foggy rows to CSV")

Saved foggy rows to CSV


In [23]:
print((~fg_in_foggy).sum())

21565


In [6]:
# # Use the weather column that exists in your dataframe
# weather_col = "HourlyPresentWeatherConditions" if "HourlyPresentWeatherConditions" in CV_rows1.columns else "HourlyPresentWeatherType"

# # Ensure we're editing a proper copy
# CV_rows1 = CV_rows1.copy()

# # Set visibility to 0.9 where weather code contains "FG" (e.g., FG, BCFG, PRFG, etc.)
# fg_mask = CV_rows1[weather_col].fillna("").astype(str).str.contains("FG", regex=False)
# CV_rows1.loc[fg_mask, "HourlyVisibility"] = 0.9

# print(f"Updated rows: {fg_mask.sum()}")

Updated rows: 852042
